In [1]:
# pakages used
# pip install yfinance pandas beautifulsoup4 requests transformers
# %pip install -U transformers torch

In [2]:

import yfinance as yf
import pandas as pd


In [3]:
import torch
print(torch.__version__)

2.13.0


In [4]:
from pprint import pprint
import requests
from bs4 import BeautifulSoup
import torch


In [5]:
import transformers
from transformers import pipeline
from transformers.utils import is_torch_available

W1019 02:12:58.096000 68956 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [6]:
search = yf.Search("AAPL") # Default gives us 8 articles
news = search.news

pprint(news[0])

df = pd.DataFrame(news)
display(df)

{'link': 'https://finance.yahoo.com/technology/articles/trump-330-million-white-house-033011715.html',
 'providerPublishTime': 1792380611,
 'publisher': 'Benzinga',
 'relatedTickers': ['NVDA', 'GOOG', 'AAPL', 'MSFT', 'PLTR'],
 'thumbnail': {'resolutions': [{'height': 1080,
                                'tag': 'original',
                                'url': 'https://media.zenfs.com/en/benzinga_79/b123636fd5270d83818c95bd0b790a36.jpg',
                                'width': 1920},
                               {'height': 140,
                                'tag': '140x140',
                                'url': 'https://s.yimg.com/lo/mysterio/api/8547D9A241C68567A86634051DAA4657D4EB55EDC78285F1A16DA33E5AEC4DE1/subgraphmysterio/resizefill_w140_h140/https:%2F%2Fmedia.zenfs.com%2Fen%2Fbenzinga_79%2Fb123636fd5270d83818c95bd0b790a36.jpg',
                                'width': 140}]},
 'title': 'Trump’s $330 Million White House Ballroom Donor List Reveals '
          'Nvidia, Appl

,uuid,title,publisher,link,providerPublishTime,type,thumbnail,relatedTickers
0,ebd65211-e0ed-3827-aa97-c5e26bdb2383,Trump’s $330 Million White House Ballroom Dono...,Benzinga,https://finance.yahoo.com/technology/articles/...,1792380611,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[NVDA, GOOG, AAPL, MSFT, PLTR]"
1,2ff978f6-f83e-3fda-b27d-b7ef37271fc5,Starboard sees big upside in Skyworks after Qo...,Investing.com,https://finance.yahoo.com/markets/stocks/artic...,1792380076,STORY,NaN,"[SWKS, QRVO, AAPL]"
2,47b7e332-b797-3be2-8898-db535a41a41a,"AAPL, TSLA, NFLX, GME, RKLB: Why Retail Trader...",Stocktwits,https://finance.yahoo.com/markets/stocks/artic...,1792375488,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[AAPL, RKLB, TSLA, GME, NFLX]"
3,98b90d94-5659-3d9b-b7d1-98079ab62fa9,"Apple’s $1,999 iPhone bet just got its first a...",TheStreet,https://finance.yahoo.com/markets/stocks/artic...,1792374420,STORY,{'resolutions': [{'url': 'https://media.zenfs....,[AAPL]
4,901cf85d-6694-3a50-9fa5-ed442eb6258b,The $5 Trillion Club: Which Is the Better Stoc...,Motley Fool,https://finance.yahoo.com/markets/stocks/artic...,1792354800,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[NVDA, AAPL]"
5,8caeb620-fc3f-393d-b19d-10721c012616,Apple Keeps Hitting Record Highs. Bitcoin Hasn...,24/7 Wall St.,https://finance.yahoo.com/markets/stocks/artic...,1792346402,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[AAPL, BTC-USD]"
6,fdd9e68b-6886-3b62-a74e-90ccc4d2e8bf,What Does Broadcom (AVGO) Face As AI Chip Supp...,Simply Wall St.,https://finance.yahoo.com/technology/ai/articl...,1792339773,STORY,{'resolutions': [{'url': 'https://media.zenfs....,"[AVGO, TSM, NVDA, AAPL]"
7,34e6da13-bbd5-3447-bdca-37413fc48bbc,Spy Games in Silicon Valley,The Wall Street Journal,https://finance.yahoo.com/m/34e6da13-bbd5-3447...,1792335300,STORY,{'resolutions': [{'url': 'https://media.zenfs....,[AAPL]


In [7]:
article = news[0]
content = article.get("content", article)

url = (content.get("canonicalUrl") or {}).get("url") or content.get("link")

print("URL:", url)

# Download article HTML
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=15
)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# Locate main article
article_html = soup.find("article")

# Extract paragraphs
if article_html:
    paragraphs = article_html.find_all("p")
    full_text = "\n\n".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
    )
else:
    full_text = None

# Display results
print("\nTITLE:")
print(content.get("title"))

print("\nFULL ARTICLE:")
print(full_text)

URL: https://finance.yahoo.com/technology/articles/trump-330-million-white-house-033011715.html

TITLE:
Trump’s $330 Million White House Ballroom Donor List Reveals Nvidia, Apple and Google Among Top Tech Contributors: Report (UPDATED)

FULL ARTICLE:
The above button links to Coinbase. Yahoo Finance is not a broker-dealer or investment adviser and does not offer securities or cryptocurrencies for sale or facilitate trading. Coinbase pays us for certain activity generated through this link. Prices displayed are informational.

Benzinga and Yahoo Finance LLC may earn commission or revenue on some items through the links below.

Editor's note: The story has been updated to include a comment from Nvidia.

President Donald Trump 's White House ballroom project donor list reportedly reveals companies and individuals linked to federal contracts, policy decisions, and a presidential pardon.

NVIDIA Corp. and Apple Inc. each pledged $25 million, matching contributions from the Adelson family an

In [8]:
# Sentiment Analysis Experiment (Unstructured -> Structured Data)

import os
os.environ["USE_TF"] = "0"

import pandas as pd
import matplotlib.pyplot as plt
from transformers import pipeline

sentiment = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    framework="pt",
    device="cpu"
)

if full_text and full_text.strip():

    results = sentiment(
        full_text,
        truncation=True,
        top_k=None
    )

    for result in results:
        print(result["label"], result["score"])

    df_sentiment = pd.DataFrame(results)

    display(df_sentiment)

else:
    print("No article text available.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

neutral 0.9015117883682251
positive 0.06799779832363129
negative 0.03049040585756302


,label,score
0,neutral,0.901512
1,positive,0.067998
2,negative,0.030490
